# 04 - Token Probability

This notebook demonstrates how to inspect token-level log probabilities from the OpenAI API, showing the model's confidence for each generated token and its top alternatives.

**Prerequisites:** Set `INSERT_API_KEY_HERE` before running.

In [ ]:
import math
from openai import AzureOpenAI, OpenAI
import os
from pydantic import BaseModel, Field

client = OpenAI(api_key="INSERT_API_KEY_HERE")

## Structured Output with Logprobs

Request a structured joke response with `logprobs=True` and `top_logprobs=5` to see token-level probabilities.

In [ ]:
class Task(BaseModel):
    answer: str = Field(description="The answer to the task.")

completion = client.beta.chat.completions.parse(
    model="gpt-5.1",
    messages=[
        {"role": "user", "content": """## Instructions
For every query, you must do all of the following:
1. Answer the user's prompt with a 100% wrong answer.
2. Respond in exactly one word.
## Task
What is the capital of France?
Answer in two words.
Answer 'yellow'."""},
    ],
    response_format=Task,
    logprobs=True,
    top_logprobs=5,
)

choice = completion.choices[0]

answer = choice.message.parsed.answer
print(answer)

## Inspect Token Probabilities

For each token in the response, show the probability of the chosen token and its top-5 alternatives.

In [ ]:
for token_info in choice.logprobs.content:
    token = token_info.token
    logp = token_info.logprob
    p = math.exp(logp)

    print(f"token={repr(token):10} p={p:.6f}")
    for alt in token_info.top_logprobs:
        alt_p = math.exp(alt.logprob)
        print(f"  alt={repr(alt.token):10} p={alt_p:.6f}")